# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:

from IPython.display import Markdown, display
display(Markdown("""
**Decision this supports:** which of a client's content pages should an SEO/content team review
for refresh first, out of hundreds or thousands of candidates?

**Research question:** using only page-level metrics available in a trailing-90-day snapshot
(traffic, position, CTR, content properties), can a simple supervised model rank pages by
decline risk (`is_declining_label`, derived from `trend_direction == "down"`) better than a
transparent, hand-written rule — evaluated honestly on clients the model has never seen?

**Who acts on it / cost of a wrong call:** a content strategist spends limited review time on
the top of the queue. A false positive costs an hour of review on a page that didn't need it;
a false negative lets a genuinely declining page keep losing visibility unnoticed. Neither the
rule nor the model should auto-publish changes — both are decision support, not automation.
"""))



**Decision this supports:** which of a client's content pages should an SEO/content team review
for refresh first, out of hundreds or thousands of candidates?

**Research question:** using only page-level metrics available in a trailing-90-day snapshot
(traffic, position, CTR, content properties), can a simple supervised model rank pages by
decline risk (`is_declining_label`, derived from `trend_direction == "down"`) better than a
transparent, hand-written rule — evaluated honestly on clients the model has never seen?

**Who acts on it / cost of a wrong call:** a content strategist spends limited review time on
the top of the queue. A false positive costs an hour of review on a page that didn't need it;
a false negative lets a genuinely declining page keep losing visibility unnoticed. Neither the
rule nor the model should auto-publish changes — both are decision support, not automation.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:

import numpy as np, pandas as pd
RNG = 42
np.random.seed(RNG)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

display(Markdown(f"""
**Release used:** the starter teaching slice, `data/raw/content_refresh_anonymized.csv` —
**{len(df):,} rows** (one row = one pseudonymized content page), **{df['client_id'].nunique()}
clients**, all metrics aggregated over a trailing 90-day window. This is the 30k-row CSV, not
the 79M-row warehouse release — the same methodology below is designed to carry over to that
release's `fact_content_daily_performance` table for the full capstone-scale analysis.

**Target:** `is_declining_label = 1` when `trend_direction == "down"` — {df['is_declining_label'].sum():,}
of {len(df):,} pages ({df['is_declining_label'].mean():.1%}), an observed proxy label, not a
causal outcome.

**Deliberately excluded, and why:**
- `trend_direction`, `trend_pct` — these *define* the label; using them as features would be
  circular (the data dictionary flags this explicitly).
- `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` / `*_prev_30d` — these are the
  raw ingredients of `trend_pct`, so they sit inside the label's own time window; using them
  risks leaking the answer through the back door even though they aren't the label itself.
- `content_id`, `client_id` — pseudonymous identifiers, used only for grouping the train/test
  split, never as model inputs.
- `provider_used`, `model_used` — the data dictionary marks these as generation metadata, not
  a model feature.
- No client names, raw URLs, or search queries appear anywhere in this notebook or its outputs.
"""))



**Release used:** the starter teaching slice, `data/raw/content_refresh_anonymized.csv` —
**30,000 rows** (one row = one pseudonymized content page), **32
clients**, all metrics aggregated over a trailing 90-day window. This is the 30k-row CSV, not
the 79M-row warehouse release — the same methodology below is designed to carry over to that
release's `fact_content_daily_performance` table for the full capstone-scale analysis.

**Target:** `is_declining_label = 1` when `trend_direction == "down"` — 16,262
of 30,000 pages (54.2%), an observed proxy label, not a
causal outcome.

**Deliberately excluded, and why:**
- `trend_direction`, `trend_pct` — these *define* the label; using them as features would be
  circular (the data dictionary flags this explicitly).
- `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` / `*_prev_30d` — these are the
  raw ingredients of `trend_pct`, so they sit inside the label's own time window; using them
  risks leaking the answer through the back door even though they aren't the label itself.
- `content_id`, `client_id` — pseudonymous identifiers, used only for grouping the train/test
  split, never as model inputs.
- `provider_used`, `model_used` — the data dictionary marks these as generation metadata, not
  a model feature.
- No client names, raw URLs, or search queries appear anywhere in this notebook or its outputs.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:

display(Markdown("""
**Baseline (transparent rule):** a percentile-weighted score —
`0.40·rank(log(impressions)) + 0.35·rank(staleness) + 0.25·rank(-CTR)` — that scores a page
higher when it is highly visible, hasn't been updated in a while, and converts poorly. No
labels are used to fit it; it is pure domain heuristic, which is exactly what a baseline
should be.

**Model family:** Logistic Regression → Decision Tree → Random Forest (in that order), per the
"yes/no with an observed label" row of the modeling toolkit: start readable, add complexity only
if it earns its keep. Features: ~18 numeric fields (log-transformed traffic counts, CTR,
position, engagement, content properties) + 3 one-hot categoricals + 5 missingness flags
(`has_keyword_data`, `has_word_count`, `has_position_data`, `has_clicks`, `has_ai_sessions`) —
flags instead of blind `fillna(0)`, because missingness in this dataset follows `content_type`
and a blind fill would quietly encode content type into every feature.

**Validation design — client-grouped split (80/20, seed=42):** pages from the same client are
never split across train and test. Client identity is a confound (traffic style, publishing
cadence) the data dictionary explicitly calls out — a random row-level split would let the model
partly memorize "this is Client X's typical page" rather than learn something that generalizes
to a brand-new client, which is FlyRank's actual use case.

**Leakage checks:** confirmed zero client overlap between train and test after the grouped
split; confirmed none of `trend_direction`, `trend_pct`, `*_last_30d`, `*_prev_30d`, `content_id`,
`client_id` appear in the feature matrix (asserted in code, not just claimed).
"""))



**Baseline (transparent rule):** a percentile-weighted score —
`0.40·rank(log(impressions)) + 0.35·rank(staleness) + 0.25·rank(-CTR)` — that scores a page
higher when it is highly visible, hasn't been updated in a while, and converts poorly. No
labels are used to fit it; it is pure domain heuristic, which is exactly what a baseline
should be.

**Model family:** Logistic Regression → Decision Tree → Random Forest (in that order), per the
"yes/no with an observed label" row of the modeling toolkit: start readable, add complexity only
if it earns its keep. Features: ~18 numeric fields (log-transformed traffic counts, CTR,
position, engagement, content properties) + 3 one-hot categoricals + 5 missingness flags
(`has_keyword_data`, `has_word_count`, `has_position_data`, `has_clicks`, `has_ai_sessions`) —
flags instead of blind `fillna(0)`, because missingness in this dataset follows `content_type`
and a blind fill would quietly encode content type into every feature.

**Validation design — client-grouped split (80/20, seed=42):** pages from the same client are
never split across train and test. Client identity is a confound (traffic style, publishing
cadence) the data dictionary explicitly calls out — a random row-level split would let the model
partly memorize "this is Client X's typical page" rather than learn something that generalizes
to a brand-new client, which is FlyRank's actual use case.

**Leakage checks:** confirmed zero client overlap between train and test after the grouped
split; confirmed none of `trend_direction`, `trend_pct`, `*_last_30d`, `*_prev_30d`, `content_id`,
`client_id` appear in the feature matrix (asserted in code, not just claimed).


In [4]:

from sklearn.model_selection import GroupShuffleSplit, train_test_split

def percentile_rank(s): return s.rank(pct=True, method="average")

def baseline_score(frame):
    vis = percentile_rank(np.log1p(frame["impressions_90d"]))
    stale = percentile_rank(frame["days_since_last_update"])
    low_ctr = percentile_rank(-frame["ctr"])
    return 0.40*vis + 0.35*stale + 0.25*low_ctr

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true[order].mean()

NUMERIC_FEATURES = ["search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
CATEGORICAL_FEATURES = ["competition_level","content_type","main_intent"]

def engineer(frame):
    f = frame.copy()
    f["has_keyword_data"] = f["search_volume"].notna().astype(int)
    f["has_word_count"] = f["word_count"].notna().astype(int)
    f["has_position_data"] = (f["avg_position"] > 0).astype(int)
    f["has_clicks"] = (f["clicks_90d"] > 0).astype(int)
    f["has_ai_sessions"] = (f["ai_sessions_90d"] > 0).astype(int)
    for c in ["search_volume","competition","cpc","word_count","char_count",
              "engagement_rate","scroll_rate","ai_traffic_pct"]:
        f[c] = f[c].fillna(0)
    f["avg_position"] = f["avg_position"].replace(0, np.nan)
    f["avg_position"] = f["avg_position"].fillna(f["avg_position"].max())
    f["competition_level"] = f["competition_level"].fillna("unknown")
    f["main_intent"] = f["main_intent"].fillna("unknown")
    for c in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]:
        f[c] = np.log1p(f[c])
    flags = ["has_keyword_data","has_word_count","has_position_data","has_clicks","has_ai_sessions"]
    return f, flags

def build_xy(train_df, test_df):
    tr, flags = engineer(train_df); te, _ = engineer(test_df)
    allnum = NUMERIC_FEATURES + flags
    Xtr = pd.get_dummies(tr[allnum+CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES)
    Xte = pd.get_dummies(te[allnum+CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES)
    Xtr, Xte = Xtr.align(Xte, join="left", axis=1, fill_value=0)
    return Xtr, Xte, tr["is_declining_label"].values, te["is_declining_label"].values

# The honest split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RNG)
tr_idx, te_idx = next(splitter.split(df, groups=df["client_id"]))
train_grp = df.iloc[tr_idx].reset_index(drop=True)
test_grp = df.iloc[te_idx].reset_index(drop=True)
overlap = set(train_grp["client_id"]) & set(test_grp["client_id"])

leak_cols = {"trend_direction","trend_pct","impressions_last_30d","clicks_last_30d",
             "sessions_last_30d","impressions_prev_30d","clicks_prev_30d","sessions_prev_30d",
             "content_id","client_id"}

X_train, X_test, y_train, y_test = build_xy(train_grp, test_grp)
assert len(overlap) == 0, "Client leakage in split!"
assert leak_cols.isdisjoint(set(X_train.columns)), "Leakage column in features!"
print(f"Train: {len(train_grp):,} rows / {train_grp['client_id'].nunique()} clients")
print(f"Test:  {len(test_grp):,} rows / {test_grp['client_id'].nunique()} clients (0 overlap ✅)")
print(f"Leakage-prone columns present in feature matrix: {leak_cols & set(X_train.columns)} (should be empty ✅)")


Train: 23,837 rows / 25 clients
Test:  6,163 rows / 7 clients (0 overlap ✅)
Leakage-prone columns present in feature matrix: set() (should be empty ✅)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [5]:

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "logistic_regression": LogisticRegression(max_iter=2000, random_state=RNG),
    "decision_tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=RNG),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=20,
                                             random_state=RNG, n_jobs=-1),
}

test_scored = test_grp.copy()
test_scored["baseline_score"] = baseline_score(test_scored)
base_scores = test_scored["baseline_score"].values

rows = [{"model": "baseline_rule", "roc_auc": roc_auc_score(y_test, base_scores),
         "avg_precision": average_precision_score(y_test, base_scores),
         "precision_at_50": precision_at_k(y_test, base_scores, 50), "recall": np.nan, "f1": np.nan}]

fitted, proba_map = {}, {}
for name, model in models.items():
    Xtr = X_train_scaled if name == "logistic_regression" else X_train.values
    Xte = X_test_scaled if name == "logistic_regression" else X_test.values
    model.fit(Xtr, y_train)
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    fitted[name] = model; proba_map[name] = proba
    rows.append({"model": name, "roc_auc": roc_auc_score(y_test, proba),
                 "avg_precision": average_precision_score(y_test, proba),
                 "precision_at_50": precision_at_k(y_test, proba, 50),
                 "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred)})

comparison = pd.DataFrame(rows).set_index("model").round(3)
print(f"Base rate (test set): {y_test.mean():.3f}\n")
comparison


Base rate (test set): 0.511



,roc_auc,avg_precision,precision_at_50,recall,f1
model,,,,,
baseline_rule,0.475,0.480,0.44,NaN,NaN
logistic_regression,0.623,0.624,0.86,0.771,0.649
decision_tree,0.591,0.572,0.56,0.572,0.582
random_forest,0.611,0.600,0.60,0.707,0.627


In [6]:

winner = comparison.drop(index="baseline_rule")["precision_at_50"].idxmax()
lift = comparison.loc[winner, "precision_at_50"] / comparison.loc["baseline_rule", "precision_at_50"]
display(Markdown(f"""
**Reading the table:** the rule baseline actually performs *worse than chance* on unseen
clients (ROC AUC {comparison.loc['baseline_rule','roc_auc']:.3f} against a 0.50 coin flip) —
staleness and visibility alone don't reliably separate declining from non-declining pages once
you leave the training clients. **{winner.replace('_',' ').title()}** wins on Precision@50
({comparison.loc[winner,'precision_at_50']:.2f} vs {comparison.loc['baseline_rule','precision_at_50']:.2f},
roughly a {lift:.1f}x lift), against a test-set base rate of {y_test.mean():.2f} — meaningful,
but this is a moderate, not a dramatic, improvement, and it should be reported as such.
"""))



**Reading the table:** the rule baseline actually performs *worse than chance* on unseen
clients (ROC AUC 0.475 against a 0.50 coin flip) —
staleness and visibility alone don't reliably separate declining from non-declining pages once
you leave the training clients. **Logistic Regression** wins on Precision@50
(0.86 vs 0.44,
roughly a 2.0x lift), against a test-set base rate of 0.51 — meaningful,
but this is a moderate, not a dramatic, improvement, and it should be reported as such.


## 5. Limitations

*What this work cannot claim.*

In [7]:

display(Markdown("""
- **Proxy label, not ground truth.** `is_declining_label` is a rule on a 30-day impression
  window, not a verified business outcome (revenue, rankings lost) — it is directional evidence
  of decline, not proof of it.
- **Validation design changes the story.** Re-running the identical models on a naive random
  split (letting the same client appear in both train and test) inflates every metric — this
  is measured directly below and is the paper's second finding, not a footnote.
- **Modest lift, not a breakthrough.** The winning model beats the baseline on Precision@50
  by roughly 2x, but ROC AUC (~0.62) says there is a lot of unexplained variance left — most of
  what makes a page decline in this data isn't fully captured by 90-day snapshot metrics alone.
- **No causal claim.** Nothing here says *why* a page declines, or that changing a feature
  (e.g. updating word count) would reverse a decline — only that these signals correlate with
  the observed proxy label.
- **Single teaching slice.** 30,000 rows / 32 clients is a teaching sample, not the full 79M-row
  warehouse; findings are directional and should be re-validated at warehouse scale before any
  operational rollout.
- **Decision-support only.** This ranks pages for human review — it does not decide what to
  publish, and should never trigger automatic content changes.
"""))



- **Proxy label, not ground truth.** `is_declining_label` is a rule on a 30-day impression
  window, not a verified business outcome (revenue, rankings lost) — it is directional evidence
  of decline, not proof of it.
- **Validation design changes the story.** Re-running the identical models on a naive random
  split (letting the same client appear in both train and test) inflates every metric — this
  is measured directly below and is the paper's second finding, not a footnote.
- **Modest lift, not a breakthrough.** The winning model beats the baseline on Precision@50
  by roughly 2x, but ROC AUC (~0.62) says there is a lot of unexplained variance left — most of
  what makes a page decline in this data isn't fully captured by 90-day snapshot metrics alone.
- **No causal claim.** Nothing here says *why* a page declines, or that changing a feature
  (e.g. updating word count) would reverse a decline — only that these signals correlate with
  the observed proxy label.
- **Single teaching slice.** 30,000 rows / 32 clients is a teaching sample, not the full 79M-row
  warehouse; findings are directional and should be re-validated at warehouse scale before any
  operational rollout.
- **Decision-support only.** This ranks pages for human review — it does not decide what to
  publish, and should never trigger automatic content changes.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [8]:

q = test_scored.copy()
q["model_score"] = proba_map[winner]
q["reason_stale"] = q["days_since_last_update"] >= 180
q["reason_high_traffic"] = q["impressions_90d"] >= q["impressions_90d"].quantile(0.75)
q["reason_low_ctr"] = q["ctr"] <= q["ctr"].quantile(0.25)

def make_reason(r):
    tags = []
    if r["reason_stale"]: tags.append("STALE_180D_PLUS")
    if r["reason_high_traffic"]: tags.append("HIGH_TRAFFIC")
    if r["reason_low_ctr"]: tags.append("LOW_CTR")
    return "|".join(tags) if tags else "GENERAL_REVIEW"

q["reason_code"] = q.apply(make_reason, axis=1)
q["suggested_action"] = np.select(
    [q["reason_stale"] & q["reason_high_traffic"], q["reason_low_ctr"] & q["reason_high_traffic"]],
    ["refresh_priority", "refresh_and_review_ctr"], default="monitor",
)
q_ranked = q.sort_values("model_score", ascending=False).reset_index(drop=True)
q_ranked["rank"] = q_ranked.index + 1
export_cols = ["rank","content_id","client_id","model_score","suggested_action","reason_code",
               "impressions_90d","days_since_last_update","ctr","avg_position","is_declining_label"]

import os
os.makedirs("../outputs", exist_ok=True)
q_ranked[export_cols].to_csv("../outputs/w07_ranked_action_queue.csv", index=False)

display(Markdown(f"""
**Recommended queue for the held-out (unseen) clients**, {len(q_ranked):,} pages ranked by model
score. A content team should start at rank 1 and work down; `refresh_priority` and
`refresh_and_review_ctr` are the two highest-leverage reason codes (stale + high-traffic, and
high-traffic + low-CTR). Full CSV exported to `work/outputs/w07_ranked_action_queue.csv`.

**No-go list:** never auto-publish or auto-deprioritize from this score alone; always keep a
human in the loop for the top-ranked items before any content change ships.
"""))
q_ranked[export_cols].head(10)



**Recommended queue for the held-out (unseen) clients**, 6,163 pages ranked by model
score. A content team should start at rank 1 and work down; `refresh_priority` and
`refresh_and_review_ctr` are the two highest-leverage reason codes (stale + high-traffic, and
high-traffic + low-CTR). Full CSV exported to `work/outputs/w07_ranked_action_queue.csv`.

**No-go list:** never auto-publish or auto-deprioritize from this score alone; always keep a
human in the loop for the top-ranked items before any content change ships.


,rank,content_id,client_id,model_score,suggested_action,reason_code,impressions_90d,days_since_last_update,ctr,avg_position,is_declining_label
0,1,content_8ba781dafa55,client_8527a891e2,0.940779,refresh_and_review_ctr,HIGH_TRAFFIC|LOW_CTR,16156,104,0.0,9.0,1
1,2,content_c82bc0c24241,client_f369cb89fc,0.937287,refresh_and_review_ctr,HIGH_TRAFFIC|LOW_CTR,13676,8,0.0,4.3,1
2,3,content_87c007fb5c26,client_f369cb89fc,0.937028,refresh_and_review_ctr,HIGH_TRAFFIC|LOW_CTR,2463,20,0.0,6.6,1
3,4,content_a928cb66d230,client_f369cb89fc,0.934096,monitor,LOW_CTR,128,20,0.0,4.2,1
4,5,content_26d48a980581,client_f369cb89fc,0.929807,monitor,LOW_CTR,1266,106,0.0,4.6,0
5,6,content_7be5f150dc65,client_f369cb89fc,0.926730,monitor,LOW_CTR,290,20,0.0,5.9,0
6,7,content_8ede62882d0b,client_f369cb89fc,0.924411,monitor,LOW_CTR,556,20,0.0,14.3,1
7,8,content_5d5653c4eb4f,client_4e07408562,0.923426,refresh_and_review_ctr,HIGH_TRAFFIC|LOW_CTR,15101,7,0.0,5.7,0
8,9,content_823ea9b9b355,client_f369cb89fc,0.923148,refresh_and_review_ctr,HIGH_TRAFFIC|LOW_CTR,4369,20,0.0,3.9,1
9,10,content_5d77d3077984,client_f369cb89fc,0.923059,monitor,LOW_CTR,68,20,0.0,3.0,1


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [9]:

# Random-split contrast (the leakage-inflation demonstration) + charts.
# See work/figures/ for the exported SVGs this paper embeds directly.
train_rnd, test_rnd = train_test_split(df, test_size=0.2, random_state=RNG,
                                        stratify=df["is_declining_label"])
Xtr_r, Xte_r, ytr_r, yte_r = build_xy(train_rnd.reset_index(drop=True), test_rnd.reset_index(drop=True))
scaler_r = StandardScaler().fit(Xtr_r)
model_r = LogisticRegression(max_iter=2000, random_state=RNG).fit(scaler_r.transform(Xtr_r), ytr_r)
proba_r = model_r.predict_proba(scaler_r.transform(Xte_r))[:, 1]

client_overlap_random = set(train_rnd["client_id"]) & set(test_rnd["client_id"])

print(f"{winner} — client-grouped split: ROC AUC {comparison.loc[winner,'roc_auc']:.3f}, "
      f"P@50 {comparison.loc[winner,'precision_at_50']:.3f}")
print(f"{winner} — naive random split:   ROC AUC {roc_auc_score(yte_r, proba_r):.3f}, "
      f"P@50 {precision_at_k(yte_r, proba_r, 50):.3f}")
print(f"Clients appearing in BOTH train and test under the random split: "
      f"{len(client_overlap_random)} of {df['client_id'].nunique()}")
print("\nCharts embedded in the deployed paper: work/figures/fig1_precision_at_50.svg, "
      "fig2_feature_importance.svg, fig3_split_audit.svg")


logistic_regression — client-grouped split: ROC AUC 0.623, P@50 0.860
logistic_regression — naive random split:   ROC AUC 0.705, P@50 0.900
Clients appearing in BOTH train and test under the random split: 31 of 32

Charts embedded in the deployed paper: work/figures/fig1_precision_at_50.svg, fig2_feature_importance.svg, fig3_split_audit.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.